# DocQuery — Retrieval-Augmented Generation Pipeline

Built from scratch: document chunking -> sentence-transformer embeddings -> FAISS index -> retrieval -> generation -> evaluation.

This notebook mirrors the local pipeline (`ingest.py`, `embed_index.py`, `retrieve.py`, `generate.py`, `evaluate.py`) but runs the real `all-MiniLM-L6-v2` embedding model, which needs internet access to download.

Compare the Recall@3 numbers at the bottom against the TF-IDF baseline noted in the final cells. That gap is the empirical justification for using dense embeddings over keyword search.


## 1. Install dependencies

In [1]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 42.7 MB/s eta 0:00:00


## 2. Sample document corpus

Embedded inline so this notebook runs with zero uploads. Swap these for your own documents by replacing the dictionary below.

In [2]:
DOCUMENTS = {
    "doc1_adversarial_robustness": "Adversarial Robustness in Neural Networks\n\nNeural networks trained on clean data can be highly sensitive to small, deliberately crafted\ninput perturbations. These perturbations, often invisible to a human observer, can cause a\nmodel to misclassify an input with high confidence. The Fast Gradient Sign Method (FGSM)\ngenerates such perturbations in a single step by moving each input pixel a small amount in\nthe direction of the gradient of the loss with respect to the input. This produces an adversarial\nexample cheaply, but the resulting perturbation is relatively weak compared to iterative methods.\n\nProjected Gradient Descent (PGD) extends this idea by taking multiple smaller gradient steps\nand projecting the result back into an allowed perturbation budget after each step. Because it\nsearches the local loss landscape more thoroughly than FGSM, PGD is considered a much\nstronger attack and is commonly used as a benchmark for adversarial training.\n\nAdversarial training incorporates these attacks directly into the training loop: at each step,\nthe model is trained not on the clean input but on an adversarially perturbed version of it.\nThis tends to improve robustness at some cost to clean accuracy, and the size of that tradeoff\ndepends heavily on the perturbation budget used during training and the capacity of the model.\n\nEvaluating robustness only on clean validation data therefore gives an incomplete picture of\nmodel reliability. A model can score well on held-out clean examples while still being\nbrittle to small, structured input changes that would never appear in curated benchmark sets\nbut frequently appear in real deployment environments, such as noisy scans, compressed\nimages, or adversarially motivated inputs from fraud actors.\n",
    "doc2_retrieval_augmented_generation": "Retrieval-Augmented Generation\n\nRetrieval-Augmented Generation, or RAG, combines a retrieval step with a generative language\nmodel to ground the model's output in an external knowledge source rather than relying solely\non what was memorized during pretraining. A typical RAG pipeline has three stages: chunking\nand embedding a document collection, retrieving the most relevant chunks for a given query,\nand passing those chunks alongside the query into a language model to produce an answer.\n\nChunking strategy matters more than it first appears. Splitting documents into fixed-size\nwindows is simple but can cut sentences or ideas in half, hurting retrieval quality. Overlapping\nwindows, where consecutive chunks share a portion of their content, reduce the chance that a\nrelevant answer is split across a chunk boundary, at the cost of a larger and more redundant\nindex.\n\nDense retrieval represents each chunk as a vector using an embedding model, typically a\nsentence-transformer trained so that semantically similar text produces vectors that are close\ntogether under cosine similarity or inner product. At query time, the query is embedded with\nthe same model and compared against every chunk vector using an approximate or exact\nnearest-neighbor search structure such as FAISS.\n\nA common failure mode in RAG systems is retrieving chunks that are lexically similar to the\nquery but not actually relevant to answering it, or missing chunks that use different wording\nfor the same concept. This is why retrieval quality should be evaluated directly, independent\nof the generation step, using a labeled set of query-to-relevant-chunk pairs rather than judging\nthe system only by whether the final generated answer looks reasonable.\n",
    "doc3_faiss_vector_search": "Vector Search with FAISS\n\nFAISS is a library for efficient similarity search over dense vectors. At small to medium scale,\nan exact search index such as IndexFlatL2 or IndexFlatIP simply compares a query vector against\nevery stored vector and returns the closest matches, guaranteeing exact nearest neighbors at the\ncost of scanning the entire index for every query. This is usually fine for collections of a few\nhundred thousand vectors or fewer.\n\nAt larger scale, exact search becomes too slow, and approximate methods trade a small amount\nof accuracy for large speed gains. IVF (inverted file) indexes partition the vector space into\nclusters using k-means, and at query time only a subset of clusters closest to the query are\nsearched, rather than the entire collection. HNSW (hierarchical navigable small world) indexes\nbuild a graph structure that allows a query to navigate toward its nearest neighbors without\nvisiting most of the index.\n\nChoosing between inner product and L2 distance depends on whether the embeddings are\nnormalized. If vectors are normalized to unit length, ranking by inner product is equivalent to\nranking by cosine similarity, and is often preferred for text embeddings because it focuses on\nthe direction of the vector rather than its magnitude.\n\nIndex quality should be validated the same way any retrieval system is validated: by measuring\nwhether known relevant items are actually returned in the top-k results for representative\nqueries, not by inspecting the index structure alone.\n",
    "doc4_prompt_injection": "Prompt Injection in Retrieval-Augmented Systems\n\nWhen a language model's context window includes text retrieved from an external source, that\nretrieved text can contain instructions rather than just information. Prompt injection occurs\nwhen a malicious or unexpected document contains language crafted to redirect the model's\nbehavior, for example instructing it to ignore its original task or reveal system-level\ninstructions. This is a distinct risk from adversarial perturbations on model inputs, because\nthe attack is expressed in natural language rather than in pixel or token-level noise.\n\nRAG systems are particularly exposed to this risk because retrieved documents are, by design,\ninserted into the model's context alongside the user's query, often without the same scrutiny\napplied to the query itself. A document scraped from an untrusted source, uploaded by another\nuser, or pulled from the open web can carry embedded instructions that the retrieval step has\nno way of filtering out, since retrieval is optimized for topical relevance, not for safety.\n\nMitigations include clearly delimiting retrieved content from instructions in the prompt\ntemplate, training or prompting the model to treat retrieved text as data rather than as\ncommands, and running a separate classifier over retrieved chunks to flag suspicious\ninstruction-like content before it reaches the generation step. None of these mitigations are\ncomplete on their own, which is why evaluation harnesses for RAG systems increasingly include\nadversarially constructed documents designed specifically to test whether injected instructions\nare followed.\n",
    "doc5_evaluation_beyond_benchmarks": "Evaluating Models Beyond Clean Benchmarks\n\nA model's accuracy on a held-out validation set answers a narrower question than it appears to:\nit tells you how the model performs on data drawn from the same distribution and curation\nprocess as the training set, not how it performs on the inputs it will actually see in\nproduction. Real deployment traffic tends to differ from benchmark data in ways that are easy\nto overlook, including compression artifacts, unusual formatting, partially corrupted inputs,\nand inputs crafted by users who are actively trying to find the model's failure modes.\n\nBuilding an evaluation harness that goes beyond clean benchmarks typically involves constructing\nor collecting examples along specific axes of difficulty rather than sampling more of the same\ndistribution. For a document understanding system this might mean testing against scanned\ndocuments with poor image quality, documents in unexpected layouts, or documents containing\nadversarially placed text. For a retrieval system it might mean testing queries phrased very\ndifferently from the wording used in the source documents.\n\nFailure analysis is most useful when it produces a specific, falsifiable hypothesis about why\nthe model failed, which can then be tested with a targeted experiment. A vague observation such\nas \"the model struggles with hard cases\" is not actionable; a hypothesis such as \"the model's\nretrieval step fails when the query uses a synonym not present in any chunk\" can be tested\ndirectly by constructing synonym-shifted queries and measuring retrieval recall specifically on\nthat subset.\n",
    "rag_systems": "Retrieval-Augmented Generation, or RAG, combines a retrieval system with a text generation model to answer questions using external documents rather than relying solely on knowledge stored in the model's parameters. This is useful because language models have a fixed knowledge cutoff and cannot cite sources for facts they generate from memory alone.\n\nA typical RAG pipeline has three stages. First, documents are split into chunks and converted into dense vector embeddings using a sentence embedding model. Second, at query time, the user's question is embedded using the same model, and a similarity search is run against the stored document embeddings to find the most relevant chunks. Third, the retrieved chunks are inserted into the generation model's prompt as context, and the model produces an answer grounded in that retrieved text.\n\nFAISS, developed by Meta, is one of the most widely used libraries for the similarity search step. It supports several index types, including flat indexes that compute exact nearest neighbors and approximate indexes such as IVF and HNSW that trade a small amount of accuracy for much faster search on large datasets.\n\nOne common failure mode in RAG systems is retrieving chunks that are semantically similar to the query but do not actually contain the answer. This can happen when a query is ambiguous, when a document uses different terminology than the query, or when a chunk boundary splits an answer away from the sentence that provides necessary context. Good RAG evaluation should specifically test these failure cases rather than only measuring performance on clean, well-matched queries.\n\nReranking is a technique used to improve retrieval quality after the initial similarity search. A reranker model, often a cross-encoder, takes each retrieved chunk together with the original query and produces a more precise relevance score than the original embedding similarity alone, at the cost of additional compute per query.\n",
    "transformers_overview": "The Transformer architecture was introduced in the 2017 paper \"Attention Is All You Need.\" Unlike earlier sequence models such as RNNs and LSTMs, transformers process all tokens in a sequence in parallel rather than one at a time. This parallelism is what makes transformers so much faster to train on modern GPU and TPU hardware.\n\nThe core mechanism inside a transformer is self-attention. Self-attention allows every token in a sequence to look at every other token and decide how much attention to pay to it when building its own representation. Each token produces three vectors: a query, a key, and a value. The attention score between two tokens is computed by taking the dot product of one token's query vector with another token's key vector, then scaling and passing through a softmax function.\n\nTransformers typically stack multiple layers of self-attention and feed-forward networks. Each layer also includes residual connections and layer normalization, which help gradients flow during training and make very deep networks trainable.\n\nPositional encoding is added to the input embeddings because self-attention itself has no inherent notion of token order. Without positional encoding, a transformer would treat a sentence as an unordered bag of tokens.\n\nLarge language models like GPT and Llama are decoder-only transformers, meaning they only use the decoder half of the original transformer architecture, with a causal masking scheme so that each token can only attend to earlier tokens in the sequence. This causal masking is what allows these models to generate text one token at a time during inference.\n\nVision Transformers, or ViTs, apply the same self-attention mechanism to images by splitting an image into fixed-size patches, flattening each patch, and treating the sequence of patches the same way a language transformer treats a sequence of word tokens.\n",
}

## 3. Ingestion and chunking

Sentence-aware chunking with overlap.

In [3]:
import re
from dataclasses import dataclass
from typing import List

@dataclass
class Chunk:
    doc_id: str
    chunk_id: int
    text: str

def split_into_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    return [s.strip() for s in sentences if s.strip()]

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> List[str]:
    sentences = split_into_sentences(text)
    chunks, current = [], ""
    for sentence in sentences:
        if len(current) + len(sentence) + 1 <= chunk_size:
            current = f"{current} {sentence}".strip()
        else:
            if current:
                chunks.append(current)
            overlap_text = current[-overlap:] if current else ""
            current = f"{overlap_text} {sentence}".strip()
    if current:
        chunks.append(current)
    return chunks

all_chunks: List[Chunk] = []
for doc_id, text in DOCUMENTS.items():
    for i, c in enumerate(chunk_text(text)):
        all_chunks.append(Chunk(doc_id=doc_id, chunk_id=i, text=c))

print(f"Total chunks: {len(all_chunks)}")
print(f"Example: {all_chunks[0]}")


Total chunks: 38
Example: Chunk(doc_id='doc1_adversarial_robustness', chunk_id=0, text='Adversarial Robustness in Neural Networks Neural networks trained on clean data can be highly sensitive to small, deliberately crafted input perturbations. These perturbations, often invisible to a human observer, can cause a model to misclassify an input with high confidence. The Fast Gradient Sign Method (FGSM) generates such perturbations in a single step by moving each input pixel a small amount in the direction of the gradient of the loss with respect to the input.')


## 4. Embed with the real sentence-transformer model

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c.text for c in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True,
                           normalize_embeddings=True).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"Indexed {index.ntotal} vectors, dim={dim}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Indexed 38 vectors, dim=384


## 5. Retrieval function

In [5]:
def retrieve(query: str, top_k: int = 3):
    query_vec = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, indices = index.search(query_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        c = all_chunks[idx]
        results.append({"doc_id": c.doc_id, "chunk_id": c.chunk_id, "text": c.text, "score": float(score)})
    return results

for r in retrieve("How does self-attention work in transformers?", top_k=2):
    print(f"[{r['score']:.3f}] {r['doc_id']} (chunk {r['chunk_id']}): {r['text'][:100]}...")


[0.681] transformers_overview (chunk 1): to train on modern GPU and TPU hardware. The core mechanism inside a transformer is self-attention. ...
[0.630] transformers_overview (chunk 0): The Transformer architecture was introduced in the 2017 paper "Attention Is All You Need." Unlike ea...


## 6. Generation (LLM mode)

Requires an API key. Store it in Colab's Secrets manager (key icon, left sidebar) as `ANTHROPIC_API_KEY` -- never paste it directly into a cell.

In [ ]:
!pip install -q anthropic

In [8]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

SYSTEM_PROMPT = (
    "You are a document question-answering assistant. Answer the user's question "
    "using ONLY the context provided below. Do not use outside knowledge. If the "
    "context does not contain enough information to answer the question, say exactly: "
    "'The retrieved context does not contain enough information to answer this question.' "
    "Cite which chunk(s) you used by their doc_id."
)

def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(f"[{c['doc_id']} | chunk {c['chunk_id']}]\n{c['text']}" for c in retrieved_chunks)
    return f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer using only the context above:"

def generate_answer(query, top_k=3):
    retrieved = retrieve(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text, retrieved

answer, sources = generate_answer("How does self-attention work in transformers?")
print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for s in sources:
    print(f"  - {s['doc_id']} (chunk {s['chunk_id']}, score {s['score']:.3f})")


ModuleNotFoundError: No module named 'anthropic'

## 7. Evaluation harness

Same three categories as the local TF-IDF test: exact-match, paraphrased, out-of-scope. Compare against the TF-IDF baseline printed at the end.

In [6]:
EXACT_MATCH_QUERIES = [
    {"query": "What is the Fast Gradient Sign Method?", "expected_doc": "doc1_adversarial_robustness"},
    {"query": "What is FAISS used for?", "expected_doc": "doc3_faiss_vector_search"},
    {"query": "What is prompt injection?", "expected_doc": "doc4_prompt_injection"},
]

PARAPHRASED_QUERIES = [
    {"query": "How can you find tiny input changes that fool a neural network?", "expected_doc": "doc1_adversarial_robustness"},
    {"query": "What tool would you use to quickly search millions of embedding vectors?", "expected_doc": "doc3_faiss_vector_search"},
    {"query": "How might someone hide malicious commands inside a document a chatbot reads?", "expected_doc": "doc4_prompt_injection"},
]

OUT_OF_SCOPE_QUERIES = [
    {"query": "What is the capital of France?"},
    {"query": "How do I bake a chocolate cake?"},
    {"query": "What was the score of yesterday's football match?"},
]

def evaluate_recall_at_k(labeled_queries, k=3, label=""):
    hits, total = 0, len(labeled_queries)
    print(f"\n=== {label} (Recall@{k}) ===")
    for item in labeled_queries:
        results = retrieve(item["query"], top_k=k)
        retrieved_docs = [r["doc_id"] for r in results]
        hit = item["expected_doc"] in retrieved_docs
        hits += int(hit)
        status = "PASS" if hit else "FAIL"
        top_score = results[0]["score"] if results else 0.0
        q = item["query"]
        top = retrieved_docs[0] if retrieved_docs else None
        print(f"  [{status}] '{q}'  top_score={top_score:.3f}  retrieved_top={top}")
    recall = hits / total if total else 0.0
    print(f"  --> Recall@{k}: {hits}/{total} = {recall:.2f}")
    return recall

def evaluate_out_of_scope(queries, threshold=0.30):
    correct, total = 0, len(queries)
    print(f"\n=== OUT-OF-SCOPE (should score below {threshold}) ===")
    for item in queries:
        results = retrieve(item["query"], top_k=1)
        top_score = results[0]["score"] if results else 0.0
        ok = top_score < threshold
        correct += int(ok)
        status = "PASS" if ok else "FAIL"
        q = item["query"]
        print(f"  [{status}] '{q}'  top_score={top_score:.3f}")
    rate = correct / total if total else 0.0
    print(f"  --> Correctly flagged as out-of-scope: {correct}/{total} = {rate:.2f}")
    return rate

exact_recall = evaluate_recall_at_k(EXACT_MATCH_QUERIES, k=3, label="EXACT-MATCH")
paraphrase_recall = evaluate_recall_at_k(PARAPHRASED_QUERIES, k=3, label="PARAPHRASED")
oos_rate = evaluate_out_of_scope(OUT_OF_SCOPE_QUERIES)

print()
print("="*50)
print("SUMMARY (real sentence-transformer embeddings)")
print("="*50)
print(f"Exact-match Recall@3:   {exact_recall:.2f}")
print(f"Paraphrased Recall@3:   {paraphrase_recall:.2f}")
print(f"Out-of-scope detection: {oos_rate:.2f}")
print()
print("TF-IDF baseline (measured locally, for comparison):")
print("  Exact-match Recall@3:   1.00")
print("  Paraphrased Recall@3:   0.67")
print("  Out-of-scope detection: 0.00")
print()
print("NOTE: the out-of-scope threshold above (0.30) is a starting guess for")
print("normalized MiniLM cosine similarity, not TF-IDF's 0.15 -- the two methods")
print("produce scores on different scales. Tune this empirically by inspecting")
print("your own score distribution for known relevant vs. irrelevant queries.")



=== EXACT-MATCH (Recall@3) ===
  [PASS] 'What is the Fast Gradient Sign Method?'  top_score=0.518  retrieved_top=doc1_adversarial_robustness
  [PASS] 'What is FAISS used for?'  top_score=0.475  retrieved_top=rag_systems
  [PASS] 'What is prompt injection?'  top_score=0.690  retrieved_top=doc4_prompt_injection
  --> Recall@3: 3/3 = 1.00

=== PARAPHRASED (Recall@3) ===
  [PASS] 'How can you find tiny input changes that fool a neural network?'  top_score=0.588  retrieved_top=doc1_adversarial_robustness
  [PASS] 'What tool would you use to quickly search millions of embedding vectors?'  top_score=0.595  retrieved_top=doc3_faiss_vector_search
  [PASS] 'How might someone hide malicious commands inside a document a chatbot reads?'  top_score=0.440  retrieved_top=doc4_prompt_injection
  --> Recall@3: 3/3 = 1.00

=== OUT-OF-SCOPE (should score below 0.3) ===
  [PASS] 'What is the capital of France?'  top_score=0.088
  [PASS] 'How do I bake a chocolate cake?'  top_score=0.074
  [PASS] 'What was

## 8. What to write down after running this

1. Actual Recall@3 for all three categories.
2. Whether the paraphrase gap shrank vs. the TF-IDF baseline, and your hypothesis for why.
3. Whether out-of-scope detection improved, and what threshold actually separated relevant from irrelevant queries in your score distribution.
4. One concrete failure case where retrieval still got it wrong, and what you'd try next (reranking, a bigger embedding model, query rewriting). This -- not the code itself -- is what "designing evaluation benchmarks beyond clean validation datasets" means in practice, and it's exactly what the JD is asking for.
